# Module 1 — Your first trace

**ODSC AI West 2026 · Production-Grade Agent & LLM Observability with Opik**

Traditional APM rests on three assumptions that LLM applications quietly violate:

1. **Determinism.** Same input, same output — so a test that passed in CI keeps passing.
   LLM calls break this on the first request.
2. **Failure is loud.** Exceptions, 5xx, timeouts. But an LLM's worst failures are its
   most well-formed ones: a hallucination is a valid string, correctly typed, returned fast.
3. **A request is one unit of work.** An agent is a tree. One duration on one endpoint
   tells you nothing about which branch was slow, expensive, or wrong.

This notebook fixes assumption 3 first, because it is the way in to the other two.

It ends with a **Your turn** exercise — time to practise on code of your own.

## Setup

You need a free [Opik account](https://www.comet.com/site/products/opik/) and an OpenAI
API key. `opik.configure()` will prompt you for the Opik key.

In [ ]:
%pip install -q opik openai

In [ ]:
import os
import getpass

import opik

PROJECT = "odsc-workshop"
# Self-hosted Opik? Export OPIK_URL_OVERRIDE before starting Jupyter (see SETUP.md).
# Otherwise this connects to Opik Cloud and asks for your API key.
SELF_HOSTED_URL = os.environ.get("OPIK_URL_OVERRIDE")
opik.configure(use_local=bool(SELF_HOSTED_URL), url_override=SELF_HOSTED_URL, project_name=PROJECT)

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (Enter to continue without one): ")

os.environ["OPIK_PROJECT_NAME"] = PROJECT

## 1. A trace is just a decorated function

No LLM yet. `@track` on a plain function already gives you the shape of everything
that follows: inputs, outputs, timing, and — crucially — nesting.

In [ ]:
from opik import track


@track
def normalize(question: str) -> str:
    return question.strip().lower()


@track
def handle(question: str) -> str:
    cleaned = normalize(question)
    return f"you asked: {cleaned}"


print(handle("  What is a SPAN?  "))

`handle` is now a **trace**. `normalize` is a **span** inside it — you did not have to
say so, the nesting was inferred from the call stack.

Follow the link Opik printed above. Note what it captured with no extra work: arguments,
return value, start and end time.

## 2. Add a model, and the trace starts carrying money

One important detail: `@track` alone does **not** record tokens or cost. The thing that
does is wrapping the client. `track_openai` patches it so every model call becomes an
LLM span carrying usage, cost, model name and provider.

In [ ]:
from openai import OpenAI
from opik.integrations.openai import track_openai

client = track_openai(OpenAI())


@track
def answer(question: str) -> str:
    reply = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
    )
    return reply.choices[0].message.content


print(answer("In one sentence: what is distributed tracing?"))

## 3. Read the trace back

The UI is the nice way to look at a trace. The SDK is the useful way, because it is how
you would build alerting, reporting, or a regression check on top of the same data.

In [ ]:
import time

from opik.rest_api.core.api_error import ApiError


def wait_for(predicate, timeout=90, interval=3):
    """Ingestion is asynchronous and batched, so a freshly logged trace is not instantly
    queryable — and until the first one lands, its project does not exist yet either.
    Poll instead of guessing at a sleep."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            found = predicate()
        except ApiError as exc:
            if exc.status_code != 404:  # 404: the project hasn't been created yet
                raise
            found = None
        if found:
            return found
        time.sleep(interval)
    raise TimeoutError(f"nothing arrived after {timeout}s")


opik.flush_tracker()
opik_client = opik.Opik()


def latest_answer():
    found = opik_client.search_traces(
        project_name=PROJECT, filter_string='name = "answer"', max_results=1
    )
    # The trace-level cost is rolled up after the spans land, so wait for it too.
    return found[0] if found and found[0].total_estimated_cost else None


latest = wait_for(latest_answer)
print(f"name     : {latest.name}")
print(f"duration : {latest.duration:.1f} ms")
print(f"cost     : ${latest.total_estimated_cost:.8f}")
print(f"spans    : {latest.span_count}")
print(f"tokens   : {(latest.usage or {}).get('total_tokens')}")

## Your turn

Practise both ideas from this module on code of your own.

1. Add a third tracked step to the pipeline — say, a `validate(question)` check that runs
   before `normalize`. Run it and open the trace: is the new span nested where you expected?
2. Tag the trace with `opik_context.update_current_trace(tags=[...])`, then find it again with
   `opik_client.search_traces(filter_string='tags contains "..."')`.

Try it first — a reference solution follows.

In [ ]:
# Your turn: add a validate() step, tag the trace, and find it again by its tag.

### Reference solution

In [ ]:
from opik import opik_context


@track
def validate(question: str) -> bool:
    return len(question.strip()) > 3


@track
def handle_v2(question: str) -> str:
    opik_context.update_current_trace(tags=["my-first-tag"])
    if not validate(question):
        return "question too short"
    return f"you asked: {normalize(question)}"


print(handle_v2("  What is a TRACE?  "))
opik.flush_tracker()

tagged = wait_for(lambda: opik_client.search_traces(
    project_name=PROJECT, filter_string='tags contains "my-first-tag"', max_results=10
))
print(f"found {len(tagged)} trace(s) tagged my-first-tag")

## What you just did

- Turned a function call into a trace with one decorator
- Got nesting for free, inferred from the call stack
- Captured tokens and cost by wrapping the client, not by writing instrumentation
- Read all of it back programmatically

That is the whole of assumption 3, solved. Next: a real agent, where the tree has
enough branches that reading it actually tells you something.

➡️ **Module 2 — `02_agent_tracing.ipynb`**